# UNIVERSIDAD ICESI
# MAESTRIA EN  IA APLICADA
# Trabajo 2
## Nombres:

#### Angelica Maria Mayor
#### Diego Agudelo
#### Freddy Mauricio Gutierrez
#### Carlos Alberto Martinez Ramirez
#### Wilman Quiñonez


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohtar10/icesi-nlp/blob/main/Sesion2/2-nlp-with-lstm.ipynb)

En este notebook implementaremos un clasificador de noticias en español utilizando la arquitectura de red LSTM. La idea es tener un punto de referencia para comparar cuando observemos la parte de transformers, por lo que utilizaremos el mismo dataset y tarea de ejemplo. Utilizarémos las utilidades de tokenización de huggingface transformers para ayudarnos con esta tarea.

El dataset contiene lo siguiente:


| Propriedad            | Descripción                                                                 |
|------------------|-----------------------------------------------------------------------------|
| **`review_id`**             |  ID único de la reseña.                          |
| **`reviewer_id`**      |   ID del usuario que dejó la reseña.                |
| **`review_title`**          |  Título de la reseña.    |
| **`review_body`**        |  Texto completo de la reseña.     |                            |
| **`star_rating`**        |  Puntuación otorgada (1–5 estrellas).      |                           |
| **`language`**        |  Idioma de la reseña (en este caso siempre "en")     |
| **`product_category`**        |  Categoría del producto (ej: libros, electrónica, etc.)      |
| **`review_date`**        |  Fecha en que se dejó la reseña.     |


📐 Tamaño aproximado El dataset completo multilingüe contiene más de 200 millones de reseñas en distintos idiomas. La versión en inglés ("en") sigue siendo enorme: millones de reseñas en varias categorías de productos.

#### Referencias

- [Long Short-Term Memory](https://www.researchgate.net/publication/13853244_Long_Short-Term_Memory#fullTextFileContent)

In [31]:
# Este comando de shell instala la librería `gensim`.
# `gensim` es una librería de Python para modelado de temas, indexación de documentos
# y recuperación de similitud con grandes corpus.
# El signo de exclamación `!` al inicio indica que este es un comando de shell
# que se ejecutará directamente en el entorno.
try:
    import gensim
except ImportError:
    print("La librería 'gensim' no está instalada. Instalándola ahora...")
    !pip install gensim
    import gensim

In [32]:
import pkg_resources
import warnings

warnings.filterwarnings('ignore')

# Obtiene una lista de los nombres de los paquetes instalados en el entorno actual.
installed_packages = [package.key for package in pkg_resources.working_set]

# Verifica si 'google-colab' está en la lista de paquetes instalados.
# Si está presente, significa que el código se está ejecutando en Google Colab.
# La variable `IN_COLAB` será True si está en Colab y False en caso contrario.
IN_COLAB = 'google-colab' in installed_packages

In [33]:
# INSTALACIÓN Y ACTUALIZACIÓN DE HERRAMIENTAS
# Actualizar el sistema y paquetes (apt-get update).
# Instalar Python 3.10 y librerías asociadas (distutils, lib2to3).
# Configurar alternativas de Python para que el intérprete por defecto.
# Instalar PyTorch Lightning y datasets.

#!test '{IN_COLAB}' = 'True' && wget  https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/requirements.txt && pip install -r requirements.txt
!test '{IN_COLAB}' = 'True' && sudo apt-get update -y
!test '{IN_COLAB}' = 'True' && sudo apt-get install python3.10 python3.10-distutils python3.10-lib2to3 -y
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.11 2
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.10 1
!test '{IN_COLAB}' = 'True' && pip install lightning datasets

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

## Dataset   

In [34]:
# Importa la función load_dataset de la librería datasets de Hugging Face.
# Esta librería es útil para cargar y trabajar con una amplia variedad de datasets,
# especialmente aquellos utilizados en procesamiento de lenguaje natural (NLP).
from datasets import load_dataset

# Carga el dataset específico "neonwatty/amazon_reviews_multi".
# El primer argumento es el nombre del dataset en el Hub de Hugging Face.
# El segundo argumento "en" especifica la configuración o subconjunto del dataset a cargar,
# en este caso, la versión en inglés.
# El tercer argumento split="train" indica que solo queremos cargar el split de entrenamiento
# del dataset. Otros splits comunes son "validation" y "test".
# El dataset cargado se asigna a la variable 'dataset'.
dataset = load_dataset("neonwatty/amazon_reviews_multi", "en", split="train")

# Imprime el objeto dataset. Esto muestra información resumida sobre el dataset cargado,
# como las características disponibles (columnas) y el número de filas (ejemplos).
# Es útil para verificar que el dataset se cargó correctamente y entender su estructura.
dataset

Dataset({
    features: ['review_title', 'review_body', 'review_id', 'stars'],
    num_rows: 200000
})

In [35]:
dataset[1]

{'review_title': 'Not use able',
 'review_body': 'the cabinet dot were all detached from backing... got me',
 'review_id': 'en_0690095',
 'stars': 1}



**Se realiza el analisis descriptivo:** Debido a que el dataset cambio mucho en comparación al que se usó en la clase, fue necesario realizar un análisis descriptivo para entender mejor el comportamiento de los textos:




In [36]:
import numpy as np

# Calculamos la longitud en tokens (palabras simples separadas por espacios)
# Itera sobre cada fila en el 'dataset',  para cada fila, accede al contenido de la columna 'review_body'.
# Divide el texto de 'review_body' en palabras usando .split() (separando por espacios).
# Calcula la longitud de la lista resultante de palabras.
# Almacena estas longitudes en una lista llamada 'text_lengths'.
text_lengths = [len(row['review_body'].split()) for row in dataset]

# Imprime la longitud del texto más corto encontrado en el dataset.
print(f"Texto más corto: {min(text_lengths)} tokens")
# Imprime la longitud del texto más largo encontrado en el dataset.
print(f"Texto más largo: {max(text_lengths)} tokens")
# Calcula la longitud promedio de los textos usando numpy.mean() y la imprime formateada a dos decimales.
print(f"Longitud promedio: {np.mean(text_lengths):.2f} tokens")

# Percentiles
# Itera sobre una lista de percentiles (50, 75, 90, 95, 99).
for p in [50, 75, 90, 95, 99]:
    # Calcula el percentil 'p' de la lista 'text_lengths' usando numpy.percentile()
    # e imprime el resultado. Esto muestra la distribución de las longitudes.
    print(f"P{p}: {np.percentile(text_lengths, p)} tokens")

Texto más corto: 1 tokens
Texto más largo: 769 tokens
Longitud promedio: 34.11 tokens
P50: 24.0 tokens
P75: 44.0 tokens
P90: 72.0 tokens
P95: 96.0 tokens
P99: 165.0 tokens


In [37]:
# Add a new column with the length of the 'review_body'
dataset = dataset.map(lambda example: {"review_body_length": len(example["review_body"].split())})

# Sort the dataset by the length of the 'review_body' column
dataset = dataset.sort("review_body_length")
dataset = dataset.remove_columns("review_body_length")
# Print the first few examples to verify the sorting
print(dataset[:5])
dataset

AttributeError: 'Dataset' object has no attribute 'reverse'

Se observa que el percentil 95 corresponde a textos con 96 tokens o menos, mientras que el percentil 99 llega hasta 165 tokens. Esto muestra que el dataset presenta variabilidad en cuanto al tamaño de los textos. Con el objetivo de abarcar la mayoría de los casos sin necesidad de realizar un recorte excesivo, se decidió utilizar un tamaño de secuencia de 128 tokens, valor que se encuentra entre ambos percentiles.

In [ ]:
import plotly.express as px

# Suponiendo que ya calculaste text_lengths así:
# text_lengths = [len(row['review_body'].split()) for row in dataset]

# Crear boxplot horizontal
fig = px.box(
    x=text_lengths,  # eje horizontal con las longitudes
    orientation="h", # orientación horizontal del boxplot
    labels={"x": "Longitud de reseña (tokens)"}, # etiqueta para el eje X
    title="Distribución de longitudes de reseñas (tokens)" # título del gráfico
)

# Mostrar el gráfico
fig.show()

# Análisis de la longitud de las reseñas

Se analizaron **200,000 reseñas**.

- **Longitud promedio:** 34 tokens  
- **Desviación estándar:** ~34 tokens → alta variabilidad  
- **Mínimo:** 1 token  
- **Máximo:** 769 tokens  

# Distribución por percentiles
- **25%** de las reseñas tienen ≤ **12 tokens**  
- **50% (mediana)** tienen ≤ **24 tokens**  
- **75%** tienen ≤ **44 tokens**  

➡️ La mayoría de los textos son relativamente cortos, aunque existe un grupo de reseñas considerablemente más largas que incrementan la media.

In [ ]:
import pandas as pd

# Convertir lista a Series de Pandas
# Convertimos la lista `text_lengths` (que contiene la longitud de cada reseña)
# a una Serie de Pandas. Una Serie es una estructura de datos unidimensional.
text_lengths_series = pd.Series(text_lengths)

# Usar describe()
# El método `.describe()` de una Serie de Pandas calcula y muestra estadísticas
# descriptivas como la cuenta total, el promedio, la desviación estándar,
# los valores mínimo y máximo, y los cuartiles (25%, 50%, 75%).
# Esto proporciona un resumen rápido de la distribución de las longitudes de las reseñas.
print(text_lengths_series.describe())

# Análisis de palabras únicas en el dataset

Al igual que con la longitud de los tokens, se buscó analizar la **cantidad de palabras únicas** presentes en el dataset.  
El objetivo es **definir un vocabulario adecuado** que minimice la aparición de palabras desconocidas en el pipeline.  

Por esta razón, se procedió a calcular y examinar la cantidad de palabras únicas contenidas en los textos.


In [ ]:
# Importa la clase Counter del módulo collections, que es una subclase de diccionario
# utilizada para contar objetos hasheables (como palabras).
from collections import Counter
# Importa la función tqdm, que sirve para mostrar una barra de progreso
# en bucles iterables. Es útil para visualizar el avance en tareas largas.
from tqdm import tqdm

# Inicializa un objeto de la clase Counter. Este objeto se comportará como un diccionario
# donde las claves serán los tokens (palabras) y los valores serán sus frecuencias.
token_counts = Counter()

# La línea 'for row in tqdm(dataset, desc="Procesando reseñas"):' itera sobre
# cada fila (o 'row') de una variable llamada 'dataset' (que debe ser un iterable
# como una lista de diccionarios o un DataFrame de pandas).
# La función 'tqdm' envuelve el iterable 'dataset' para generar una barra de progreso.
# El parámetro 'desc' añade una descripción a la barra de progreso, en este caso,
# "Procesando reseñas".
for row in tqdm(dataset, desc="Procesando reseñas"):
    # Concatena las cadenas de texto asociadas a las claves 'review_title' y 'review_body'
    # de cada fila, las une con un espacio y convierte todo a minúsculas para
    # asegurar que las palabras como "Bueno" y "bueno" se cuenten como el mismo token.
    text = (row['review_title'] + " " + row['review_body']).lower()

    # Divide la cadena de texto 'text' en una lista de palabras (tokens) utilizando
    # los espacios en blanco como delimitadores. Esto crea una lista de palabras.
    tokens = text.split()

    # Actualiza el objeto Counter 'token_counts' con los nuevos tokens encontrados.
    # El método 'update' incrementa el recuento de los tokens que ya existen
    # y añade los nuevos con un recuento inicial de 1.
    token_counts.update(tokens)

# La función len() se usa para obtener el número de elementos en el objeto Counter.
# En este caso, devuelve el número de claves únicas, que corresponde al número
# total de palabras únicas que se encontraron en todo el dataset.
num_unique_words = len(token_counts)

# Imprime una cadena de texto que incluye el valor de 'num_unique_words',
# mostrando el recuento total de palabras únicas.
print("Palabras únicas en el dataset:", num_unique_words)

# Imprime una cadena de texto para introducir la siguiente línea de salida.
print("Ejemplos de tokens y sus recuentos:")

# La función list() convierte el objeto 'token_counts.items()' en una lista.
# El método .items() devuelve una vista de los pares (token, recuento).
# El slicing '[:20]' limita la lista a los primeros 20 pares.
# Esto es útil para mostrar un ejemplo del contenido del contador sin imprimir todo.
print(list(token_counts.items())[:20])


## Tokenizador

Se decidió trabajar con un **vocabulario de 40,000 palabras**, construido a partir de las reseñas.  
Este tamaño permite cubrir la gran mayoría de los textos y, al mismo tiempo, controlar la complejidad del modelo.  

Además, se reservaron dos tokens especiales:

- **[PAD] (padding):** utilizado para rellenar las secuencias más cortas, de manera que todas tengan la misma longitud.  
- **[UNK] (unknown):** representa las palabras fuera del vocabulario (OOV) que no se encuentran dentro de las 40,000 más frecuentes.  

In [ ]:
import re
from collections import Counter
from tqdm import tqdm  # Importa tqdm para mostrar progreso

def simple_tokenizer(text):
    # Convierte todo el texto a minúsculas para asegurar consistencia.
    text = text.lower()
    # Usa una expresión regular para encontrar secuencias de letras (incluyendo acentos y 'ñ')
    # y reemplazar cualquier otro carácter con un espacio. Esto elimina puntuación, números, etc.
    text = re.sub(r"[^a-záéíóúüñ]+", " ", text)
    # Elimina espacios al inicio y al final del texto resultante, y luego divide
    # el texto en una lista de palabras (tokens) usando los espacios como delimitadores.
    return text.strip().split()

# Construimos el vocabulario a partir de conjunto de datos.
token_counts = Counter() # Inicializa un contador para llevar el recuento de cada token.
for text in tqdm(dataset["review_body"], desc="Procesando reseñas"): # Itera sobre el texto de la columna 'review_body' en el dataset.
    token_counts.update(simple_tokenizer(text)) # Tokeniza el texto de cada reseña y actualiza el contador.

# 40k-2 porque necesitamos reservar espacio para los dos tokens especiales
# Obtiene los tokens más frecuentes del contador y los almacena en una lista.
# Limita la lista a los 40000 tokens más frecuentes, reservando 2 espacios para [PAD] y [UNK].
top_n_tokens = list(token_counts.keys())[:40000-2]
# Inicializa el diccionario de vocabulario con los tokens especiales [PAD] y [UNK]
# asignándoles los IDs 0 y 1 respectivamente.
vocab = {"[PAD]": 0, "[UNK]": 1}
# Itera sobre los tokens más frecuentes.
for token in tqdm(top_n_tokens, desc="Procesando tokens"):
    vocab[token] = len(vocab) # Asigna un ID secuencial a cada token, empezando después de los especiales.

# Define una función para tokenizar un texto dado y convertirlo a una secuencia de IDs.
def tokenize_text(text, max_length=50):
    tokens = simple_tokenizer(text) # Tokeniza el texto de entrada.
    # Convierte cada token a su ID correspondiente usando el diccionario 'vocab'.
    # Si un token no está en el vocabulario (es una palabra desconocida), le asigna el ID de [UNK].
    # Trunca la lista de IDs si excede la 'max_length'.
    ids = [vocab.get(tok, vocab["[UNK]"]) for tok in tokens[:max_length]]
    # Rellena la lista de IDs con el ID de [PAD] hasta alcanzar la 'max_length'.
    ids += [vocab["[PAD]"]] * (max_length - len(ids))
    return ids # Devuelve la lista de IDs (representación numérica del texto).

In [ ]:
print(f"Vocabulario: {len(vocab)} tokens")
print("Primeros 15 tokens:")
print(f"{top_n_tokens[:15]}")
print("15 tokens de en medio:")
print(f"{top_n_tokens[1000:1015]}")
print("Últimos 15 tokens:")
print(f"{top_n_tokens[-15:]}")

In [ ]:
# Llama a la función `tokenize_text` con el texto de ejemplo "hello active"
# y especifica una longitud máxima de secuencia de 8.
# La función tokenizará el texto, lo convertirá a una secuencia de IDs numéricos
# basados en el vocabulario preconstruido y aplicará relleno ([PAD]) si es necesario
# para alcanzar la longitud máxima especificada.
tokenized = tokenize_text("hello active", max_length=8)
# Imprime la lista resultante de IDs numéricos.
# Esto muestra la representación numérica del texto de entrada después de la tokenización y el relleno.
tokenized

In [ ]:
from tqdm import tqdm  # Importa tqdm para mostrar progreso
# Crea un diccionario que mapea los IDs de los tokens a los tokens (palabras).
# Esto se hace invirtiendo el diccionario `vocab`, donde las claves eran los tokens
# y los valores eran sus IDs. Ahora las claves son los IDs y los valores son los tokens.
id_2_token = {v: k for k, v in tqdm(vocab.items(), desc="Creando id_2_token")}

# Convierte la secuencia de IDs tokenizados de nuevo a tokens (palabras)
# utilizando el diccionario `id_2_token`.
# Itera sobre cada ID en la lista `tokenized` (la salida de la función `tokenize_text`).
# Para cada ID, busca el token correspondiente en el diccionario `id_2_token`.
# Esto resulta en una lista de tokens (palabras) que representan la secuencia tokenizada original.
[id_2_token[token] for token in tokenized]

## Pytorch

In [ ]:
import numpy as np

# Usa numpy.unique para encontrar los valores únicos en la columna 'stars' del dataset.
# Esto nos da la lista de posibles calificaciones de estrellas (ej. [1, 2, 3, 4, 5]).
# enumerate() asigna un índice numérico (empezando desde 0) a cada valor único.
# dict() convierte este resultado en un diccionario donde las claves son los índices
# (0, 1, 2, 3, 4) y los valores son las calificaciones de estrellas correspondientes
# (1, 2, 3, 4, 5). Este diccionario mapea un ID numérico a una clase de estrella.
dict(enumerate(np.unique(dataset[:]['stars'])))

Se estableció un **seq_length de 128 tokens**, decisión basada en el análisis descriptivo de la distribución de longitudes. Este valor permite cubrir la gran mayoría de los textos sin necesidad de realizar un recorte excesivo, manteniendo un buen balance entre **cobertura** y **eficiencia computacional**.

Este código prepara reseñas de Amazon para modelos de PyTorch, convirtiendo el texto en secuencias de tokens y las estrellas en etiquetas numéricas.
Esto es lo que necesitas para luego entrenar una LSTM

In [ ]:
import torch
import numpy as np
from typing import Tuple, Dict
from torch.utils.data import Dataset

class amazon_dataset(Dataset):
    # El constructor inicializa el dataset personalizado.
    # Recibe un tokenizador (función), el dataset original de Hugging Face y la longitud máxima de secuencia.
    def __init__(self, tokenizer, dataset, seq_length: int = 128):
        # Almacena el tokenizador, el dataset original y la longitud máxima.
        self.tokenizer = tokenizer
        self.dataset = dataset
        self.seq_length = seq_length
        # Definimos estos dos mapas para facilitarnos la tarea
        # de traducir de nombres de categoría a ids de categoría.
        # id_2_class_map: Mapea IDs numéricos (0, 1, 2, 3, 4) a las clases originales de estrellas (1, 2, 3, 4, 5).
        self.id_2_class_map = dict(enumerate(np.unique(dataset[:]['stars'])))
        # class_2_id_map: Mapea las clases originales de estrellas (1, 2, 3, 4, 5) a IDs numéricos (0, 1, 2, 3, 4).
        self.class_2_id_map = {v: k for k, v in self.id_2_class_map.items()}
        # Almacena el número total de clases (5 en este caso).
        self.num_classes = len(self.id_2_class_map)

    # Este método es fundamental para PyTorch Datasets y define cómo obtener un elemento individual.
    # Recibe un índice y devuelve un diccionario con los datos de entrada y la etiqueta.
    def __getitem__(self, index) -> Dict[str, torch.Tensor]:
        # Obtiene el texto de la reseña y la calificación de estrellas del dataset original por índice.
        text, y = self.dataset[index]['review_body'], self.dataset[index]['stars']
        # Convierte la calificación de estrellas a su ID numérico usando el mapa.
        y = self.class_2_id_map[y]
        # Tokeniza el texto de la reseña, limita la longitud y lo convierte a un tensor de PyTorch.
        data = {'input_ids': torch.tensor(self.tokenizer(text, max_length=self.seq_length))}
        # Convierte el ID numérico de la etiqueta a un tensor de PyTorch y lo añade al diccionario de datos.
        data['y'] = torch.tensor(y)
        # Devuelve el diccionario con los datos preparados.
        return data

    # Este método es requerido para PyTorch Datasets y devuelve el número total de elementos en el dataset.
    def __len__(self):
        # Retorna la longitud del dataset original.
        return len(self.dataset)

In [ ]:
max_len = 128
# Instancia la clase `amazon_dataset`.
# Pasa la función `tokenize_text` como el tokenizador,
# el objeto `dataset` de Hugging Face como los datos crudos,
# y `max_len` como la longitud máxima de la secuencia.
data_amazon = amazon_dataset(tokenize_text, dataset, seq_length=max_len)
# Afirma (verifica) que la longitud de la instancia `data_amazon`
# es igual a la longitud del dataset original. Si no son iguales,
# el programa se detendrá con un AssertionError.
assert len(data_amazon) == len(dataset)

In [ ]:
data_amazon[1]

Este código toma el dataset de reseñas de Amazon (data_amazon), lo divide en conjuntos de entrenamiento, validación y prueba de forma estratificada, y genera los correspondientes DataLoaders que se utilizan para alimentar el modelo durante las fases de entrenamiento, validación y evaluación. Lo cual lo hace estratificado para no tener ningun incovenientes de balanceo.

In [ ]:
# Importa la función train_test_split para dividir los datos de forma estratificada.
from sklearn.model_selection import train_test_split

# Importa Subset y DataLoader de PyTorch.
# Subset permite seleccionar un subconjunto de un dataset,
# y DataLoader facilita la carga en batches y el manejo eficiente de los datos.
from torch.utils.data import Subset, DataLoader

# ========================================
# Paso 1: Extraer las etiquetas del dataset
# ========================================
# Suponemos que 'data_amazon' es un objeto Dataset personalizado compatible con PyTorch,
# donde cada elemento es un diccionario que contiene una clave 'y' con la etiqueta.
# Aquí se construye una lista con todas las etiquetas del dataset,
# convirtiéndolas a enteros si no lo son ya.
labels = [int(data_amazon[i]['y']) for i in range(len(data_amazon))]
# Crea una lista con los índices de todos los elementos del dataset.
# Esto se usará para dividir el dataset sin modificar su contenido directamente.
idx = list(range(len(data_amazon)))

# ========================================
# Paso 2: Dividir en conjunto de entrenamiento y conjunto temporal (validación + test)
# ========================================
# Divide los índices (idx) en dos grupos:
# - idx_train: índices para el conjunto de entrenamiento
# - idx_temp: índices para un conjunto temporal que más adelante se dividirá en validación y test
# El argumento `stratify=labels` asegura que la proporción de clases se mantenga igual en ambas divisiones.
# random_state se fija para asegurar reproducibilidad.
print("Divide en conjunto de entrenamiento y conjunto temporal")
idx_train, idx_temp, y_train, y_temp = train_test_split(
    idx, labels, test_size=0.2, random_state=42, stratify=labels
)

# ========================================
# Paso 3: Dividir el conjunto temporal en validación y test
# ========================================

# Se divide el conjunto temporal (idx_temp) en dos partes iguales:
# - idx_val: conjunto de validación
# - idx_test: conjunto de prueba
# Se vuelve a usar estratificación con las etiquetas temporales (y_temp) para mantener balance de clases.
print("Divide el conjunto temporal (idx_temp) en dos partes iguales")
idx_val, idx_test, _, _ = train_test_split(
    idx_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# ========================================
# Paso 4: Crear Subsets a partir de los índices
# ========================================
# Crea subconjuntos del dataset original usando los índices calculados en pasos anteriores.
# Estos subconjuntos son objetos Dataset que PyTorch puede usar directamente.
print("Train:", len(idx_train), "Val:", len(idx_val), "Test:", len(idx_test))
print("Creando Subsets Train")
train_dataset = Subset(data_amazon, idx_train)
print("Creando Subsets Val")
val_dataset   = Subset(data_amazon, idx_val)
print("Creando Subsets Test")
test_dataset  = Subset(data_amazon, idx_test)

# ========================================
# Paso 5: Crear DataLoaders para PyTorch
# ========================================
# Define el tamaño de batch dinámicamente.
# Si estás en Google Colab (mayor RAM/GPU), puedes usar un batch grande (128).
# En otro entorno más limitado, usar un batch pequeño (4) puede evitar cuelgues o errores de memoria.
batch_size = 128 if IN_COLAB else 4

# Crea el DataLoader para el conjunto de entrenamiento.
# shuffle=True para mezclar los datos en cada época y mejorar el entrenamiento.
print("Creando DataLoader de Train")
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=2)

# Crea los DataLoaders para validación y test.
# shuffle=False ya que no se necesita mezclar datos en estas fases (no hay entrenamiento).
print("Creando DataLoader de Val")
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=2)
print("Creando DataLoader de Test")
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=2)


In [ ]:
# Obtiene el primer batch del DataLoader de entrenamiento.
# `iter(train_loader)` crea un iterador sobre el DataLoader.
# `next(...)` obtiene el siguiente (en este caso, el primer) elemento del iterador.
# El batch es un diccionario que contiene los tensores de entrada ('input_ids') y las etiquetas ('y').
b = next(iter(train_loader))

# Imprime información sobre el tensor de IDs de entrada.
# `b["input_ids"].shape` muestra la forma del tensor (ej: [tamaño_batch, longitud_secuencia]).
# `b["input_ids"].dtype` muestra el tipo de datos del tensor (ej: torch.int64).
print("input_ids:", b["input_ids"].shape, b["input_ids"].dtype)   # [B, L], torch.int64

# Imprime información sobre el tensor de etiquetas.
# `b["y"].shape` muestra la forma del tensor (ej: [tamaño_batch]).
# `b["y"].dtype` muestra el tipo de datos del tensor (ej: torch.int64).
# `b["y"].min().item()` obtiene el valor mínimo de las etiquetas en el batch.
# `b["y"].max().item()` obtiene el valor máximo de las etiquetas en el batch.
# Esto ayuda a verificar el rango de las etiquetas (de 0 a num_clases - 1).
print("y:", b["y"].shape, b["y"].dtype, b["y"].min().item(), b["y"].max().item())  # [B], int64, 0..4

In [ ]:
train_loader.dataset[0]

### Exploración inicial: Modelo BASELINE
En un primer momento, se utilizó la misma arquitectura de red neuronal presentada en clase.  
A pesar de realizar múltiples iteraciones y ajustes, el rendimiento no mejoró de manera significativa, obteniéndose una **precisión aproximada de 0.20**.  

Adicionalmente, trabajar en **Google Colab** representó una limitación, ya que el uso de GPU está restringido y dificulta la ejecución de múltiples corridas.  

Con el objetivo de determinar si ese era realmente el **máximo rendimiento alcanzable con la LSTM** o si existían oportunidades de mejora, se decidió entrenar también **modelos de Machine Learning tradicionales**.  
Esto permitió establecer una **línea base de comparación** y evaluar con mayor claridad el potencial de mejora de la arquitectura LSTM.



[Modelo de multinomial-TF-IDF](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from tqdm import tqdm  # Importa tqdm para mostrar progreso

# Esta función auxiliar desciende a través de atributos '.dataset'
# para encontrar el dataset base original (por ejemplo, un Hugging Face Dataset)
# que contiene los datos brutos de texto.
def _unwrap_base_dataset(ds):
    """Desciende por atributos .dataset hasta llegar al dataset base (HF Dataset)."""
    base = ds
    # ds puede ser tu CustomDataset envuelto por Subset; vamos bajando mientras exista .dataset
    while hasattr(base, "dataset"):
        base = base.dataset
    return base

# Esta función convierte un objeto torch.utils.data.Subset
# en listas de texto (X) y etiquetas numéricas (y).
# Utiliza los índices del Subset para acceder a los datos originales
# en el dataset base y extrae el texto (título + cuerpo) y la etiqueta (ya mapeada a 0-4).
def subset_to_xy_from_base(subset, max_n=None, use_title=True):
    """
    Convierte un torch.utils.data.Subset -> (X, y) usando:
      - y: del propio subset[i]['y'] (ya en 0..4)
      - texto: del dataset base (HF) vía subset.indices -> row['review_title']/'review_body'].
    """
    # 1) verifica si es un Subset válido con índices
    if not hasattr(subset, "indices"):
        raise ValueError("El objeto pasado no es un Subset o no tiene .indices")

    # Accede a las capas inferiores del dataset para obtener el dataset base
    base_custom = subset.dataset            # tu CustomDataset
    base_source = _unwrap_base_dataset(base_custom)  # HF Dataset u otro contenedor con texto

    # Obtiene los índices globales a los que apunta el Subset
    idxs = subset.indices
    # Determina cuántos ejemplos procesar (todos o un máximo especificado)
    n = len(idxs) if max_n is None else min(max_n, len(idxs))

    X, y = [], [] # Listas para almacenar el texto y las etiquetas
    # Itera sobre los índices seleccionados en el Subset
    for i in tqdm(range(n), desc="Procesando"):
        # Obtiene el índice global del ejemplo en el dataset original
        idx_global = idxs[i] # Índice global en el dataset original

        # La etiqueta ya está mapeada a 0..4 en tu CustomDataset
        y_i = int(subset[i]["y"])

        # Obtiene la fila del dataset base usando el índice global
        row = base_source[idx_global]
        # Extrae título y cuerpo, manejando posibles valores None y convirtiendo a string vacío si es necesario
        title = str(row.get("review_title", "") or "")
        body  = str(row.get("review_body", "")  or "")
        # Combina título y cuerpo con un punto si use_title es True, de lo contrario solo usa el cuerpo
        text  = (title + ". " + body).strip() if use_title else body.strip()
        # Fallback por si el dataset usa una clave genérica 'text' en lugar de 'review_body'
        if not text:
            text = str(row.get("text", "") or "")

        X.append(text) # Añade el texto a la lista de features (X)
        y.append(y_i)  # Añade la etiqueta a la lista de labels (y)
    return X, y # Devuelve las listas de texto y etiquetas

# ---- Construye X,y para train/val/test ----
# Extrae los datos de texto (X) y las etiquetas numéricas (y)
# para los conjuntos de entrenamiento, validación y prueba
print("Ejecutar subset_to_xy_from_base para Train")
X_train, y_train = subset_to_xy_from_base(train_dataset)   # puedes pasar max_n=20000 para ir más rápido
print("Ejecutar subset_to_xy_from_base para Val")
X_val,   y_val   = subset_to_xy_from_base(val_dataset)
print("Ejecutar subset_to_xy_from_base para Test")
X_test,  y_test  = subset_to_xy_from_base(test_dataset)

# Imprime un ejemplo del primer elemento y los tamaños de los conjuntos
print("Ejemplo:", X_train[0][:120], "...", y_train[0])
print("Tamaños:", len(X_train), len(X_val), len(X_test))

# ---- TF-IDF ----
# Inicializa un vectorizador TF-IDF.
print("Vectorizando textos con TF-IDF...")
vec = TfidfVectorizer(
    max_features=50_000,    # max_features: limita el vocabulario a las 50000 palabras/n-gramas más frecuentes.
    ngram_range=(1, 2),     # ngram_range: considera unigramas (palabras individuales) y bigramas (pares de palabras).
    min_df=2,               # min_df: ignora términos que aparecen en menos de 2 documentos.
    strip_accents="unicode",# strip_accents: elimina acentos durante el preprocesamiento.
    lowercase=True,          # lowercase: convierte todo el texto a minúsculas.
    #stop_words="english",    # stop_words: elimina palabras comunes (stop words) en inglés.
    sublinear_tf=True        # sublinear_tf: escala logarítmicamente las frecuencias de términos (TF).
)
# Ajusta (fit) el vectorizador a los datos de entrenamiento y transforma (transform)
# el texto de entrenamiento en una matriz TF-IDF dispersa (Xtr).
print("Ajustando TF-IDF...")
Xtr = vec.fit_transform(X_train)
# Transforma los datos de validación usando el vectorizador ajustado en el entrenamiento.
print("Transformando TF-IDF...")
Xva = vec.transform(X_val)

# ---- Clasificador (CPU, rápido) ----
# Inicializa un modelo de Regresión Logística para clasificación multiclase.
print("Entrenando modelo de Regresión Logística...")
clf = LogisticRegression(
    max_iter=100,             # max_iter: número máximo de iteraciones para el solver.
    multi_class="ovr",        # multi_class: especifica que es un problema de clasificación multiclase.
    solver="lbfgs",           # solver: algoritmo para optimizar el modelo.
    n_jobs=-1                 # n_jobs: número de núcleos de CPU a usar (-1 usa todos disponibles).
)
# Entrena el modelo de Regresión Logística con los datos de entrenamiento vectorizados y las etiquetas.
print("Ajustando Regresión Logística...")
clf.fit(Xtr, y_train)

# ---- Métricas ----
# Realiza predicciones en el conjunto de validación.
print("Evaluando en validación...")
pred_val = clf.predict(Xva)
# Calcula la precisión (accuracy) en el conjunto de validación.
acc_val = accuracy_score(y_val, pred_val)
# Imprime la precisión de validación.
print(f"\nTF-IDF + Logistic Regression | Val Accuracy: {acc_val:.4f}\n")
# Imprime un reporte completo de clasificación (precisión, recall, F1-score) por clase.
print(classification_report(y_val, pred_val, digits=4))

# Muestra la matriz de confusión para visualizar el rendimiento por clase.
ConfusionMatrixDisplay.from_predictions(y_val, pred_val)
plt.title("Confusion Matrix - Validation") # Título de la matriz de confusión
plt.tight_layout() # Ajusta el diseño para evitar solapamientos
plt.show() # Muestra el gráfico

Se entrenó un modelo TF-IDF + Regresión Logística con 160k reseñas de entrenamiento y se evaluó en 20k de validación. El modelo logró una accuracy del 60.5%.

El mejor desempeño está en la clase 4 estrellas (precisión 0.74, recall 0.79, f1=0.76).

También destaca la clase 0 estrellas con f1≈0.68.

Las clases 1 y 2 estrellas son las más difíciles de distinguir (f1≈0.49).

En general, el modelo diferencia bien opiniones muy negativas o muy positivas, pero confunde las reseñas de valoración intermedia.

Modelo Lightgbm-TF-IDF

In [ ]:
# pip install lightgbm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from lightgbm import LGBMClassifier
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm  # Barra de progreso

# ================= Helpers para extraer X,y de tus Subset =================
def _unwrap_base_dataset(ds):
    """Desciende por atributos .dataset hasta llegar al dataset base (HF Dataset)."""
    base = ds
    while hasattr(base, "dataset"):
        base = base.dataset
    return base

def subset_to_xy_from_base(subset, max_n=None, use_title=True):
    """
    Convierte un torch.utils.data.Subset -> (X, y).
      y: subset[i]['y'] (0..C-1)
      texto: del dataset base (HF) vía subset.indices -> 'review_title' + 'review_body'
    """
    if not hasattr(subset, "indices"):
        raise ValueError("El objeto pasado no es un Subset o no tiene .indices")

    base_custom = subset.dataset
    base_source = _unwrap_base_dataset(base_custom)

    idxs = subset.indices
    n = len(idxs) if max_n is None else min(max_n, len(idxs))

    X, y = [], []

    for i in tqdm(range(n), desc="Procesando"):
        idx_global = idxs[i]
        y_i = int(subset[i]["y"])
        row = base_source[idx_global]
        title = str(row.get("review_title", "") or "")
        body  = str(row.get("review_body", "")  or "")
        text  = (title + ". " + body).strip() if use_title else body.strip()
        if not text:
            text = str(row.get("text", "") or "")
        X.append(text)
        y.append(y_i)
    return X, y

# ================= Construcción de X,y (mismo flujo que tu código) =================
print("Extrayendo datos del conjunto de entrenamiento...")
X_train, y_train = subset_to_xy_from_base(train_dataset)
print("Extrayendo datos del conjunto de validación...")
X_val,   y_val   = subset_to_xy_from_base(val_dataset)
print("Extrayendo datos del conjunto de test...")
X_test,  y_test  = subset_to_xy_from_base(test_dataset)

print("Ejemplo:", X_train[0][:120], "...", y_train[0])
print("Tamaños:", len(X_train), len(X_val), len(X_test))

# ================= TF-IDF (igual que antes) =================
print("Vectorizando textos con TF-IDF...")
vec = TfidfVectorizer(
    max_features=50_000,          # Número máximo de características (tokens) a conservar según puntuación TF-IDF
    ngram_range=(1, 2),           # Considera unigramas (1 palabra) y bigramas (2 palabras)
    min_df=2,                     # Ignora términos que aparecen en menos de 2 documentos
    strip_accents="unicode",      # Elimina acentos (normaliza caracteres acentuados a ASCII)
    lowercase=True,                # Convierte todo el texto a minúsculas antes de tokenizar
    #stop_words="english",          # Elimina palabras comunes (stop words) en inglés
    sublinear_tf=True            # Aplica escala logarítmica a las frecuencias de términos (TF)
)
Xtr = vec.fit_transform(X_train)   # CSR
Xva = vec.transform(X_val)
Xte = vec.transform(X_test)

# ================= LightGBM (multiclase) =================
print("Entrenando modelo LightGBM...")
num_classes = len(np.unique(y_train))
clf = LGBMClassifier(
    objective="multiclass",      # Tipo de tarea: clasificación multiclase
    num_class=num_classes,       # Número de clases a predecir (opcional si se puede inferir del y)

    n_estimators=800,            # Número de árboles (iteraciones de boosting). Más árboles pueden mejorar el ajuste, pero aumentan el tiempo de entrenamiento.
    learning_rate=0.05,          # Tasa de aprendizaje (shrinkage). Controla cuánto contribuye cada árbol; valores más bajos requieren más árboles.

    max_depth=-1,                # Profundidad máxima de los árboles. -1 significa sin límite, lo que permite crecimiento total.
    num_leaves=63,               # Número máximo de hojas por árbol. Controla la complejidad del modelo; más hojas = mayor capacidad.

    subsample=0.9,               # Proporción de muestras usadas para cada árbol (bagging_fraction). Ayuda a reducir overfitting.
    colsample_bytree=0.9,        # Proporción de características usadas por árbol (feature_fraction). También combate el overfitting.

    reg_lambda=1.0,              # Regularización L2. Penaliza grandes pesos en las hojas del árbol; previene sobreajuste.
    n_jobs=-1,                    # Número de núcleos para paralelización. -1 usa todos los disponibles en la máquina.
    #force_col_wise=True, # ← Fuerza procesamiento column-wise para mejorar rendimiento
    #verbosity=-1, # para silenciar advertencias innecesarias
    #min_child_samples=30, # valor por defecto 20; se sube a 30–50 para regularizar
    #min_split_gain=0.01, # agrega umbral a la ganancia de información
    #max_bin=max_len # default; reducirlo (por ej. a 128) acelera a cambio de precisión
)

# ================= Entrenamiento con early stopping (callbacks) =================
# Compatible con LightGBM >= 4.0
print("Ajustando LightGBM...")
clf.fit(
    Xtr, y_train,                            # Datos de entrenamiento: características (Xtr) y etiquetas (y_train)

    eval_set=[(Xva, y_val)],                # Conjunto de validación para evaluar el modelo durante el entrenamiento

    eval_metric=["multi_logloss", "multi_error"],
    # Métricas de evaluación durante el entrenamiento:
    # - "multi_logloss": log-loss para clasificación multiclase (métrica principal para clasificación probabilística)
    # - "multi_error": tasa de error (1 - accuracy)

    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        # Detiene el entrenamiento si no hay mejora en la métrica de validación en 50 iteraciones consecutivas

        lgb.log_evaluation(period=50)
        # Imprime métricas de evaluación cada 50 iteraciones
    ]
)

# ================= Métricas en validación =================
print("Evaluando en validación...")
pred_val = clf.predict(Xva)
acc_val = accuracy_score(y_val, pred_val)
print(f"\nTF-IDF + LightGBM | Val Accuracy: {acc_val:.4f}\n")
print(classification_report(y_val, pred_val, digits=4))

ConfusionMatrixDisplay.from_predictions(y_val, pred_val)
plt.title("Confusion Matrix - Validation (LightGBM)")
plt.tight_layout()
plt.show()

# ================= (Opcional) Evaluación en test =================
print("Evaluando en test...")
pred_test = clf.predict(Xte)
acc_test = accuracy_score(y_test, pred_test)
print(f"TF-IDF + LightGBM | Test Accuracy: {acc_test:.4f}")


Modelo de multinomial-Word2vec

In [ ]:
import re
import numpy as np
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from tqdm import tqdm  # Barra de progreso

# ================= Helpers idénticos =================
def _unwrap_base_dataset(ds):
    base = ds
    while hasattr(base, "dataset"):
        base = base.dataset
    return base

def subset_to_xy_from_base(subset, max_n=None, use_title=True):
    if not hasattr(subset, "indices"):
        raise ValueError("El objeto pasado no es un Subset o no tiene .indices")
    base_custom = subset.dataset
    base_source = _unwrap_base_dataset(base_custom)
    idxs = subset.indices
    n = len(idxs) if max_n is None else min(max_n, len(idxs))
    X, y = [], []
    for i in tqdm(range(n), desc="Procesando subset"):
        idx_global = idxs[i]
        y_i = int(subset[i]["y"])
        row = base_source[idx_global]
        title = str(row.get("review_title", "") or "")
        body  = str(row.get("review_body", "")  or "")
        text  = (title + ". " + body).strip() if use_title else body.strip()
        if not text:
            text = str(row.get("text", "") or "")
        X.append(text)
        y.append(y_i)
    return X, y

# ================= Carga datos =================
# Carga los datos de texto (X) y las etiquetas numéricas (y)
# para los conjuntos de entrenamiento, validación y prueba utilizando la función auxiliar.
X_train, y_train = subset_to_xy_from_base(train_dataset)
X_val,   y_val   = subset_to_xy_from_base(val_dataset)
X_test,  y_test  = subset_to_xy_from_base(test_dataset)

# ================= Tokenizador =================
# Define una expresión regular para encontrar tokens (secuencias de letras/números y apóstrofes opcionales).
_token_re = re.compile(r"[a-záéíóúüñ0-9]+(?:'[a-z0-9]+)?", re.IGNORECASE)
# Define una función para tokenizar un string dado, convirtiéndolo a minúsculas
# y encontrando todos los patrones que coinciden con la expresión regular.
def tokenize(s: str):
    return _token_re.findall(s.lower())

# Tokeniza el texto del conjunto de entrenamiento para preparar los datos para Word2Vec.
sentences_tr = [tokenize(t) for t in X_train]

# ================= Word2Vec =================
# Inicializa y entrena un modelo Word2Vec.
w2v = Word2Vec(
    sentences=sentences_tr,  # Datos de entrenamiento tokenizados.
    vector_size=300,         # Dimensionalidad de los vectores de palabras.
    window=5,                # Tamaño de la ventana de contexto.
    min_count=2,             # Ignora palabras con una frecuencia menor a 2.
    workers=4,               # Número de hilos de trabajo para el entrenamiento.
    sg=1,                    # Usa el algoritmo Skip-gram (generalmente mejor para tareas de embeddings).
    negative=10,             # Número de ejemplos negativos para Negative Sampling.
    epochs=8,                # Número de iteraciones (épocas) sobre el corpus.
    seed=42,                 # Semilla para la reproducibilidad.
)

# ================= texto -> vector (promedio simple) =================
# Define una función para obtener el vector promedio de un documento (lista de tokens).
def doc_vector(tokens, model: Word2Vec):
    kv = model.wv # Accede a los vectores de palabras del modelo entrenado.
    # Obtiene los vectores para cada token si el token está en el vocabulario del modelo.
    vecs = [kv[tok] for tok in tokens if tok in kv]
    # Si no hay vectores (ningún token en el vocabulario), devuelve un vector de ceros.
    if not vecs:
        return np.zeros(model.vector_size, dtype=np.float32)
    # Calcula el promedio de los vectores de los tokens.
    return np.mean(vecs, axis=0)

# Define una función para convertir una lista de textos en una matriz de vectores de documentos.
def texts_to_matrix(texts, model: Word2Vec, batch_size=4096):
    # Inicializa una matriz de ceros para almacenar los vectores de los documentos.
    out = np.zeros((len(texts), model.vector_size), dtype=np.float32)
    i = 0
    # Procesa los textos en lotes (batches).
    while i < len(texts):
        j = min(i + batch_size, len(texts))
        # Tokeniza el lote actual de textos.
        toks_batch = [tokenize(t) for t in texts[i:j]]
        # Calcula el vector de documento para cada texto en el lote.
        for k, toks in enumerate(toks_batch):
            out[i + k] = doc_vector(toks, model)
        i = j # Avanza al siguiente lote.
    return out # Devuelve la matriz de vectores de documentos.

# ================= Vectorizar =================
# Convierte los conjuntos de entrenamiento, validación y prueba en matrices de vectores
# utilizando el modelo Word2Vec entrenado.
Xtr = texts_to_matrix(X_train, w2v)
Xva = texts_to_matrix(X_val,   w2v)
Xte = texts_to_matrix(X_test,  w2v)

# Imprime las dimensiones de las matrices de embeddings generadas.
print("Embeddings shape:", Xtr.shape, Xva.shape, Xte.shape)

# ================= Clasificador =================
# Inicializa un modelo de Regresión Logística para clasificación multiclase.
clf = LogisticRegression(
    max_iter=500,             # Número máximo de iteraciones para el optimizador.
    multi_class="multinomial",# Configura para clasificación multiclase.
    solver="lbfgs",           # Algoritmo de optimización.
    n_jobs=-1,                # Usa todos los núcleos de CPU disponibles.
)
# Entrena el modelo de Regresión Logística con los vectores de documentos de entrenamiento
# y sus etiquetas correspondientes.
clf.fit(Xtr, y_train)

# ================= Evaluación =================
# Realiza predicciones en el conjunto de validación.
pred_val = clf.predict(Xva)
# Calcula la precisión en el conjunto de validación.
acc_val = accuracy_score(y_val, pred_val)
# Imprime la precisión de validación.
print(f"\nWord2Vec(mean) + Logistic Regression | Val Accuracy: {acc_val:.4f}\n")
# Imprime un reporte completo de clasificación (precisión, recall, F1-score) por clase.
print(classification_report(y_val, pred_val, digits=4))

# Muestra la matriz de confusión para visualizar el rendimiento por clase.
ConfusionMatrixDisplay.from_predictions(y_val, pred_val)
plt.title("Confusion Matrix - Validation (Word2Vec + LR)") # Título de la matriz de confusión.
plt.tight_layout() # Ajusta el diseño para evitar solapamientos.
plt.show() # Muestra el gráfico.

# Realiza predicciones en el conjunto de prueba (opcional).
pred_test = clf.predict(Xte)
# Imprime la precisión en el conjunto de prueba.
print("Test Accuracy:", accuracy_score(y_test, pred_test))

Modelo lightgbm-Word2vec

In [ ]:
# pip install gensim lightgbm
# Este comando de shell instala las librerías necesarias para el proyecto:
# - `gensim`: para el modelado de temas y embeddings de palabras (Word2Vec).
# - `lightgbm`: para el clasificador de boosting de árboles, que es muy eficiente.

# ================= Importaciones =================
import re
import numpy as np
from gensim.models import Word2Vec
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from lightgbm import LGBMClassifier
import lightgbm as lgb
import matplotlib.pyplot as plt
from tqdm import tqdm  # Barra de progreso

# ================= Funciones de Ayuda para la Extracción de Datos =================

def _unwrap_base_dataset(ds):
    """
    Función auxiliar para "desenvolver" un objeto de tipo Subset (o un objeto anidado)
    hasta llegar al dataset base de Hugging Face (HF) o a la fuente de datos original.
    Esto es necesario porque `Subset` envuelve el dataset, y a veces la estructura
    de datos puede estar anidada, haciendo que el acceso directo sea complicado.
    """
    base = ds
    while hasattr(base, "dataset"):
        base = base.dataset
    return base

def subset_to_xy_from_base(subset, max_n=None, use_title=True):
    """
    Convierte un objeto de tipo `torch.utils.data.Subset` en dos listas:
    `X` (una lista de textos) y `y` (una lista de etiquetas).

    Parámetros:
      - subset: El objeto `Subset` que contiene los índices a procesar.
      - max_n: El número máximo de ejemplos a procesar. Si es `None`, procesa todos.
      - use_title: Si es `True`, concatena el título y el cuerpo del texto.
                   Si es `False`, solo usa el cuerpo.

    El proceso es el siguiente:
    1. Verifica si el objeto de entrada es un `Subset` con el atributo `.indices`.
    2. Obtiene el dataset base (original) usando la función `_unwrap_base_dataset`.
    3. Itera sobre los índices del `Subset`.
    4. Para cada índice, extrae la etiqueta `y` y el texto de la reseña, combinando
       el título y el cuerpo si `use_title` es `True`.
    5. Agrega el texto y la etiqueta a las listas `X` y `y`, respectivamente.
    6. Muestra una barra de progreso usando `tqdm` durante el procesamiento.
    """
    if not hasattr(subset, "indices"):
        raise ValueError("El objeto pasado no es un Subset o no tiene .indices")

    base_custom = subset.dataset
    base_source = _unwrap_base_dataset(base_custom)

    idxs = subset.indices
    n = len(idxs) if max_n is None else min(max_n, len(idxs))

    X, y = [], []
    for i in tqdm(range(n), desc="Procesando subset"):
        idx_global = idxs[i]
        y_i = int(subset[i]["y"])
        row = base_source[idx_global]
        title = str(row.get("review_title", "") or "")
        body  = str(row.get("review_body", "")  or "")
        text  = (title + ". " + body).strip() if use_title else body.strip()
        if not text:
            text = str(row.get("text", "") or "")
        X.append(text)
        y.append(y_i)
    return X, y

# ================= Construcción de los Conjuntos de Datos =================

# Llama a la función `subset_to_xy_from_base` para generar las listas
# de textos (X) y etiquetas (y) para los conjuntos de entrenamiento,
# validación y prueba a partir de los `Subset` de PyTorch.
X_train, y_train = subset_to_xy_from_base(train_dataset)
X_val,   y_val   = subset_to_xy_from_base(val_dataset)
X_test,  y_test  = subset_to_xy_from_base(test_dataset)

# Imprime un ejemplo para verificar que los datos se extrajeron correctamente.
print("Ejemplo:", X_train[0][:120], "...", y_train[0])
# Imprime el tamaño de cada conjunto para verificar la división.
print("Tamaños:", len(X_train), len(X_val), len(X_test))

# ================= Tokenizador Simple =================

# Define una expresión regular (regex) para encontrar tokens.
# `r"[a-záéíóúüñ0-9]+(?:'[a-z0-9]+)?"` busca palabras compuestas por letras
# del español y números, incluyendo contracciones como "o'clock".
_token_re = re.compile(r"[a-záéíóúüñ0-9]+(?:'[a-z0-9]+)?", re.IGNORECASE)

def tokenize(s: str):
    """
    Función que toma una cadena de texto y la divide en una lista de tokens.
    Convierte la cadena a minúsculas y utiliza la expresión regular precompilada
    para encontrar todos los tokens que coincidan con el patrón.
    """
    return _token_re.findall(s.lower())

# Aplica la función `tokenize` a cada texto en el conjunto de entrenamiento.
# El resultado es una lista de listas, donde cada sublista contiene los tokens
# de una reseña. Esto es lo que `Word2Vec` espera como entrada.
sentences_tr = [tokenize(t) for t in X_train]

# ================= Entrenamiento del Modelo Word2Vec =================

# Entrena el modelo Word2Vec de gensim. Word2Vec aprende a representar palabras
# como vectores numéricos densos (embeddings) basándose en su contexto.
# Esto captura las relaciones semánticas entre las palabras.
w2v = Word2Vec(
    sentences=sentences_tr,  # Datos de entrenamiento tokenizados
    vector_size=300,         # Dimensión de los vectores de salida (embeddings).
    window=5,                # Tamaño de la ventana de contexto.
    min_count=2,             # Ignora palabras con frecuencia menor a 2.
    workers=4,               # Número de hilos para el entrenamiento.
    sg=1,                    # Algoritmo: 1 para Skip-gram, 0 para CBOW. Skip-gram suele ser más lento pero mejor para conjuntos pequeños.
    negative=10,             # Número de muestras negativas para el entrenamiento.
    epochs=8,                # Número de iteraciones sobre los datos.
    seed=42,                 # Semilla para la reproducibilidad del entrenamiento.
)

# ================= Vectorización de Textos =================

def doc_vector(tokens, model: Word2Vec):
    """
    Convierte una lista de tokens en un único vector, promediando los
    vectores de las palabras que están en el vocabulario del modelo.
    Si ningún token está en el vocabulario, devuelve un vector de ceros.
    Este es un método de agregación de embeddings simple pero efectivo.
    """
    kv = model.wv  # Acceso a los vectores de palabras (KeyedVectors) del modelo entrenado.
    vecs = [kv[t] for t in tokens if t in kv]
    if not vecs:
        return np.zeros(model.vector_size, dtype=np.float32)
    return np.mean(vecs, axis=0)

def texts_to_matrix(texts, model: Word2Vec, batch_size=4096):
    """
    Aplica la función `doc_vector` a una lista de textos para convertirlos
    en una matriz NumPy de vectores.

    - texts: Lista de cadenas de texto a vectorizar.
    - model: El modelo Word2Vec entrenado.
    - batch_size: Procesa los textos en lotes para reducir el uso de memoria.

    El resultado es una matriz donde cada fila es el vector de una reseña,
    y el número de columnas es la dimensión de los vectores (`vector_size`).
    """
    out = np.zeros((len(texts), model.vector_size), dtype=np.float32)
    i = 0
    while i < len(texts):
        j = min(i + batch_size, len(texts))
        toks_batch = [tokenize(t) for t in texts[i:j]]
        for k, toks in enumerate(toks_batch):
            out[i + k] = doc_vector(toks, model)
        i = j
    return out

# ================= Proceso de Vectorización =================

print("TRAIN:", len(X_train), "VAL:", len(X_val), "TEST:", len(X_test))
print("Tamaño de los vectores:", w2v.vector_size)
print("Tokens en vocabulario:", len(w2v.wv.key_to_index))
print("Vectorizando textos con Word2Vec...")

# Llama a `texts_to_matrix` para transformar los conjuntos de entrenamiento,
# validación y prueba en matrices numéricas. Estas matrices son la entrada
# para el modelo de clasificación.
print("Vectorizando textos con Word2Vec Train")
Xtr = texts_to_matrix(X_train, w2v)
print("Vectorizando textos con Word2Vec Val")
Xva = texts_to_matrix(X_val,   w2v)
print("Vectorizando textos con Word2Vec Test")
Xte = texts_to_matrix(X_test,  w2v)

# Imprime las dimensiones de las matrices resultantes.
print("Embeddings shape:", Xtr.shape, Xva.shape, Xte.shape)

# ================= Entrenamiento del Clasificador LightGBM =================

# Crea una instancia del clasificador LightGBM (LGBMClassifier).
# Es un modelo de boosting de gradiente que utiliza árboles de decisión
# para hacer la clasificación. Es muy rápido y eficiente.
num_classes = len(np.unique(y_train))
clf = LGBMClassifier(
    objective="multiclass",  # Indica que es un problema de clasificación multiclase.
    num_class=num_classes,   # Número de clases a predecir.
    n_estimators=800,        # Número de árboles que se construirán.
    learning_rate=0.05,      # Tasa de aprendizaje.
    max_depth=-1,            # Sin límite en la profundidad de los árboles.
    num_leaves=63,           # Número máximo de hojas en cada árbol.
    subsample=0.9,           # Proporción de datos para el muestreo de bagging.
    colsample_bytree=0.9,    # Proporción de características para el muestreo en cada árbol.
    reg_lambda=1.0,          # L2 regularización.
    n_jobs=-1,               # Usa todos los núcleos disponibles de la CPU.
)

# ================= Proceso de Entrenamiento y Early Stopping =================

# Entrena el clasificador con los datos vectorizados.
# `fit` entrena el modelo usando los datos de entrenamiento (Xtr, y_train).
clf.fit(
    Xtr, y_train,
    eval_set=[(Xva, y_val)],  # Los datos de validación se usan para monitorear el rendimiento.
    eval_metric=["multi_logloss", "multi_error"], # Métricas a monitorear.
    callbacks=[
        lgb.early_stopping(stopping_rounds=50), # Detiene el entrenamiento si el rendimiento
                                                # en el conjunto de validación no mejora
                                                # durante 50 rondas. Esto evita el sobreajuste.
        lgb.log_evaluation(period=50) # Imprime las métricas de evaluación cada 50 rondas.
    ]
)

# ================= Evaluación en el Conjunto de Validación =================

# Usa el modelo entrenado para predecir las etiquetas del conjunto de validación.
pred_val = clf.predict(Xva)
# Calcula la precisión del modelo en el conjunto de validación.
acc_val = accuracy_score(y_val, pred_val)
print(f"\nWord2Vec(mean) + LightGBM | Val Accuracy: {acc_val:.4f}\n")
# Imprime un reporte completo de clasificación (precisión, recall, F1-score) para
# cada clase, lo que da una visión más detallada del rendimiento.
print(classification_report(y_val, pred_val, digits=4))

# Genera y muestra una matriz de confusión para visualizar el rendimiento por clase.
# Es una herramienta visual que muestra dónde el modelo acierta y dónde se equivoca.
ConfusionMatrixDisplay.from_predictions(y_val, pred_val)
plt.title("Confusion Matrix - Validation (Word2Vec + LightGBM)")
plt.tight_layout()
plt.show()

# ================= Evaluación Final en el Conjunto de Prueba =================

# (Opcional) Evalúa el rendimiento del modelo en el conjunto de prueba,
# que nunca se usó durante el entrenamiento ni el ajuste de hiperparámetros.
# Este es un paso crítico para obtener una métrica de rendimiento no sesgada.
pred_test = clf.predict(Xte)
acc_test = accuracy_score(y_test, pred_test)
print(f"Word2Vec(mean) + LightGBM | Test Accuracy: {acc_test:.4f}")


## LSTM

La siguiente  arquitectura LSTM funciona bien para clasificar reseñas porque es potente pero sencilla de entrenar: usar varias capas con dropout ayuda a que el modelo generalice mejor y, si se activa el modo bidireccional, puede captar relaciones en ambos sentidos del texto. Además, tomar hidden[-1] como salida es práctico y confiable, ya que resume toda la información de la secuencia en un solo vector que luego se usa fácilmente para clasificar.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LSTMBlock(nn.Module):
    # El constructor inicializa las capas de la red.
    # Recibe el tamaño del vocabulario, la dimensión de los embeddings,
    # la dimensión de los estados ocultos del LSTM, el número de clases,
    # el número de capas del LSTM (por defecto 2) y la tasa de dropout (por defecto 0.2).
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers=2, dropout=0.2):
        super().__init__() # Llama al constructor de la clase padre (nn.Module).
        # Capa de Embedding: Convierte los IDs de los tokens en vectores densos.
        # vocab_size: Número total de tokens únicos.
        # embed_dim: Dimensión de los vectores de embedding.
        # padding_idx=0: El token con ID 0 (usualmente PAD) tendrá un vector de ceros y no se actualizará.
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # Capa LSTM: Procesa secuencias de vectores.
        # embed_dim: Tamaño de las características de entrada (la dimensión de los embeddings).
        # hidden_dim: Número de características en el estado oculto (y de celda).
        # num_layers: Número de capas LSTM apiladas.
        # batch_first=True: Indica que la dimensión del batch es la primera (Batch, Secuencia, Características).
        # dropout=dropout: Aplica dropout a las salidas de las capas LSTM (excepto la última) para regularización.
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)

    # Define el flujo de la información hacia adelante a través del bloque.
    # Recibe 'x', un tensor con los IDs de los tokens de entrada (shape [Batch Size, Sequence Length]).
    def forward(self, x):
        # Pasa los IDs de entrada por la capa de embedding para obtener sus representaciones vectoriales.
        embedded = self.embedding(x) # shape [Batch Size, Sequence Length, embed_dim]
        # Pasa los embeddings por la capa LSTM.
        # 'output' es la salida del último layer para cada paso de tiempo.
        # '(hidden, _)' contiene el estado oculto final (hidden) y el estado de celda final (_) para cada capa.
        output, (hidden, _) = self.lstm(embedded)
        # Retorna el estado oculto del último layer del LSTM al final de la secuencia.
        # Esto se usa como una representación vectorial fija de la secuencia de entrada.
        # hidden tiene shape [num_layers * num_directions, Batch Size, hidden_dim].
        # hidden[-1] accede al estado oculto del último layer para todos los ejemplos del batch.
        return hidden[-1]

Este modelo usa una arquitectura LSTM + MLP que convierte cada reseña en una representación fija mediante el último estado oculto y luego la clasifica con capas densas y dropout. Se entrena con PyTorch Lightning, usando CrossEntropyLoss, Accuracy y AdamW para optimización. Es una configuración robusta porque combina la capacidad del LSTM para capturar dependencias secuenciales con regularización y un pipeline de entrenamiento estable

In [ ]:
# Importa clases de PyTorch Lightning y torchmetrics.
from pytorch_lightning import LightningModule, Trainer
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from torchmetrics import Accuracy

class comments_classifier(LightningModule):
    # Hereda de `LightningModule`, la clase central de PyTorch Lightning.
    def __init__(self, vocab_size: int, num_classes: int, emb_dim: int, hidden_dim: int = 128):
        # El constructor del modelo.
        super(comments_classifier, self).__init__()
        self.num_classes = num_classes # Almacena el número de clases.

        # 1. Instancia del bloque LSTM para el procesamiento del texto.
        self.lstm = LSTMBlock(vocab_size, emb_dim, hidden_dim, num_classes)

        # 2. Capas de clasificación (red feed-forward).
        # Estas capas toman el vector de salida del LSTM y lo mapean al número de clases.
        self.classifier = nn.Sequential(
            nn.Flatten(), # Aplanar el tensor (en este caso, no es estrictamente necesario, pero es buena práctica).
            nn.Linear(hidden_dim, 128), # Capa lineal de entrada a 128 neuronas.
            nn.ReLU(),                  # Función de activación ReLU.
            nn.Dropout(0.2),            # Dropout para regularización.
            nn.Linear(128, 64),         # Capa lineal a 64 neuronas.
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes), # Capa de salida con el número de neuronas igual al de clases.
        )

        # 3. Métricas de precisión usando `torchmetrics`.
        # Se crean instancias separadas para entrenamiento, validación y prueba para que las métricas
        # se calculen de manera independiente para cada fase.
        self.train_acc = Accuracy(task='multiclass', num_classes=num_classes)
        self.val_acc = Accuracy(task='multiclass', num_classes=num_classes)
        self.test_acc = Accuracy(task='multiclass', num_classes=num_classes)

    def forward(self, x):
        # Define el paso de adelante de todo el modelo.
        # Primero, el tensor pasa por el bloque LSTM para obtener los embeddings del texto.
        embeddings = self.lstm(x)
        # Luego, estos embeddings se pasan a la red de clasificación.
        return self.classifier(embeddings)

    def training_step(self, batch, batch_idx):
        # Define la lógica de un solo paso de entrenamiento.
        # `batch` contiene los datos y las etiquetas.
        x, y = batch['input_ids'], batch['y']
        y_hat = self(x) # Realiza una predicción.
        loss = F.cross_entropy(y_hat, y) # Calcula la pérdida (Cross-Entropy Loss es estándar para clasificación).

        # 4. Registro de métricas y pérdida.
        # `self.train_acc(y_hat, y)` actualiza la métrica de precisión de entrenamiento.
        # `self.log` registra los valores para visualizarlos en TensorBoard.
        self.train_acc(y_hat, y)
        self.log('train-loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log('train-acc', self.train_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch):
        # Define la lógica de un solo paso de validación. Es similar al de entrenamiento,
        # pero no se calcula el gradiente, y las métricas se registran por separado.
        x, y = batch['input_ids'], batch['y']
        y_hat = self(x)
        loss = F.cross_entropy(y_hat, y)
        self.val_acc(y_hat, y)
        self.log('val-loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log('val-acc', self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def test_step(self, batch):
        # Define la lógica de un solo paso de prueba. Similar a la validación,
        # pero para el conjunto de prueba final.
        x, y = batch['input_ids'], batch['y']
        y_hat = self(x)
        self.test_acc(y_hat, y)
        self.log('test-acc', self.test_acc, prog_bar=True, on_step=False, on_epoch=True)

    def predict_step(self, batch):
        # Define la lógica para la inferencia (predicción) en nuevos datos.
        x = batch['input_ids']
        return self(x)

    def configure_optimizers(self):
        # Define el optimizador a utilizar. Se llama una sola vez antes de entrenar.
        # `AdamW` es una variante de Adam que es muy efectiva para modelos de aprendizaje profundo.
        optimizer =  torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-5)
        return optimizer

# ================= Configuración y Entrenamiento del Modelo =================
# 1. Instanciación del modelo.
# Se crea un objeto de la clase `comments_classifier` con los parámetros definidos.
# `vocab_size=len(vocab) + 1`: El tamaño del vocabulario, más 1 para el token de padding.
model = comments_classifier(vocab_size=len(vocab) + 1, num_classes=5, emb_dim=32)

# 2. Configuración del logger y los callbacks.
# `TensorBoardLogger`: Registra los datos de entrenamiento y validación para visualización.
# `EarlyStopping`: Un callback que detiene el entrenamiento si la métrica monitoreada
# (`train-loss` en este caso) deja de mejorar.
tb_logger = TensorBoardLogger('tb_logs', name='LSTMClassifier')
callbacks=[EarlyStopping(monitor='train-loss', patience=3, mode='min')]

# 3. Configuración del `Trainer`.
use_gpu = torch.cuda.is_available() # `use_gpu`: Comprueba si una GPU está disponible para acelerar el entrenamiento.
trainer = Trainer(
    accelerator="gpu" if use_gpu else "cpu", # `accelerator`: Selecciona entre "gpu" y "cpu".
    devices=1,
    precision="16-mixed" if use_gpu else 32, # `precision`: Usa precisión mixta de 16 bits para un entrenamiento más rápido.
    max_epochs=1, # `max_epochs`: Limita el número de épocas.
    logger=tb_logger, # `logger`: Asigna el logger a la instancia del trainer.
    callbacks=callbacks # `callbacks`: Asigna el callback de early stopping.

)

# 4. Inicio del entrenamiento.
# El método `fit` inicia el bucle de entrenamiento, utilizando los DataLoaders
# (que deben estar definidos en otra parte del código) para cargar los datos.
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)


In [ ]:
# Obtiene el primer batch del DataLoader de entrenamiento.
# `iter(train_loader)` crea un iterador sobre el DataLoader.
# `next(...)` obtiene el siguiente (en este caso, el primer) elemento del iterador.
# El batch es un diccionario que contiene los tensores de entrada ('input_ids') y las etiquetas ('y').
b = next(iter(train_loader))

# Imprime información sobre el tensor de IDs de entrada.
# `b["input_ids"].shape` muestra la forma del tensor (ej: [tamaño_batch, longitud_secuencia]).
# `b["input_ids"].dtype` muestra el tipo de datos del tensor (ej: torch.int64).
print("input_ids:", b["input_ids"].shape, b["input_ids"].dtype)   # [B, L], torch.int64

# Imprime información sobre el tensor de etiquetas.
# `b["y"].shape` muestra la forma del tensor (ej: [tamaño_batch]).
# `b["y"].dtype` muestra el tipo de datos del tensor (ej: torch.int64).
# `b["y"].min().item()` obtiene el valor mínimo de las etiquetas en el batch.
# `b["y"].max().item()` obtiene el valor máximo de las etiquetas en el batch.
# Esto ayuda a verificar el rango de las etiquetas (de 0 a num_clases - 1).
print("y:", b["y"].shape, b["y"].dtype, b["y"].min().item(), b["y"].max().item())  # [B], int64, 0..4

In [ ]:
with torch.no_grad():
    out = model(b["input_ids"])
print("logits:", out.shape, out.dtype, out.min().item(), out.max().item())  # [B, 5]

In [ ]:
from torch.utils.data import Subset, DataLoader
import torch
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping

# 1) subset pequeño
small_train = Subset(train_dataset, range(500))
small_val   = Subset(val_dataset, range(200))

sl = DataLoader(small_train, batch_size=64, shuffle=True)
sv = DataLoader(small_val,   batch_size=64, shuffle=False)

# 2) trainer rápido
use_gpu = torch.cuda.is_available()
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping

trainer = Trainer(
    max_epochs=8,     # Número máximo de épocas (pasadas completas por el conjunto de entrenamiento)
    accelerator="gpu" if use_gpu else "cpu",      # Dispositivo de aceleración: "gpu" si está disponible, si no, "cpu"
    devices=1,      # Número de dispositivos (GPUs o CPUs) a usar. Aquí solo se utiliza uno.
    precision="16-mixed" if use_gpu else 32,      # Precisión numérica:
    # - "16-mixed" usa entrenamiento de precisión mixta (más rápido y menos memoria en GPU)
    # - 32 es la precisión estándar en CPU
    callbacks=[
        EarlyStopping( # Evita el sobreajuste y reduce tiempo de entrenamiento si no hay mejora.
            monitor='val-loss',      # Métrica a monitorear
            patience=2,              # Número de épocas sin mejora antes de detener
            mode='min'               # El objetivo es minimizar la métrica (val-loss)
        )
    ],
    enable_progress_bar=True
    # Muestra barra de progreso durante el entrenamiento
)


# 3) ENTRENAR
trainer.fit(model, train_dataloaders=sl, val_dataloaders=sv)


Este código mide qué tan representativo es tu vocabulario y cuántas palabras de las reseñas quedaron fuera

In [ ]:
pad_id = vocab["[PAD]"]
unk_id = vocab["[UNK]"]

def stats_batch(batch):
    ids = batch["input_ids"]         # Tensor [B, L] con los IDs de tokens del batch
    real = (ids != pad_id)           # Máscara de tokens "reales" (no PAD)
    unk  = (ids == unk_id) & real     # Máscara de tokens <unk>, excluyendo PADs
    real_tokens = real.sum().item()    # Total de tokens reales (sin PAD)
    unk_tokens  = unk.sum().item()    # Total de tokens desconocidos (UNK) dentro de los reales
    print(f"%%UNK entre tokens no-PAD: {unk_tokens / max(1,real_tokens):.2%}")  # Imprime el porcentaje de <unk> entre los tokens válidos, evita división por cero

b = next(iter(train_loader))
stats_batch(b)

Se evidencia que ninguna palabra de las reseñas se quedaron por fuera en el entrenamiento.

In [ ]:
def pad_ratio_batch(batch):
    ids = batch["input_ids"]  # Tensor [B, L] con los tokens del batch
    pad_ratio = (ids == pad_id).float().mean().item() # Proporción de PADs en todo el batch
    print(f"PAD ratio (promedio en el batch): {pad_ratio:.2%}")

pad_ratio_batch(b)


Este código mide qué tan distintos son los embeddings que genera el LSTM para cada ejemplo del batch.

In [ ]:
with torch.no_grad():
    feats = model.lstm(b["input_ids"].to(model.device))  # [B, H]
    diffs = (feats[1:] - feats[:-1]).abs().mean(dim=1).cpu().numpy()
    print("Dif media entre ejemplos consecutivos (primer batch):", diffs.mean())

Ese resultado significa que, en tu primer batch, las representaciones que genera el LSTM para ejemplos consecutivos son muy parecidas: la diferencia promedio es apenas 0.0028 (en un rango donde los valores de los embeddings suelen estar en torno a 0.1–1.0).

In [ ]:
print(model.classifier)

Este bloque es la parte final del modelo, la que hace la clasificación.
Primero aplana la salida del LSTM, luego pasa por varias capas densas con ReLU y Dropout (para aprender mejor y evitar sobreajuste), y al final saca un vector de tamaño 5, que corresponde a las clases posibles de las reseñas.

In [ ]:
# ================== Imports necesarios ==================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import EarlyStopping
from torchmetrics import Accuracy

# (opcional) pequeñas mejoras de rendimiento
torch.backends.cudnn.benchmark = True

# ================== Bloque LSTM con packing ==================
class LSTMBlockPacked(nn.Module):
    def __init__(
        self,
        vocab_size,               # Tamaño del vocabulario (número total de tokens únicos)
        embed_dim=128,            # Dimensión de los embeddings (vectores que representan palabras)
        hidden_dim=128,           # Dimensión del estado oculto del LSTM
        num_layers=1,             # Número de capas LSTM apiladas
        dropout=0.2,              # Dropout entre capas LSTM (solo si num_layers > 1)
        padding_idx=0,            # Índice del token de padding (para ignorarlo en los embeddings)
        bidirectional=True,       # Si usar LSTM bidireccional (captura contexto hacia adelante y atrás)
        emb_dropout=0.1           # Dropout aplicado a la capa de embeddings (regularización temprana)
    ):
        super().__init__()
        self.bidirectional = bidirectional

        # Capa de embeddings (convierte índices de palabras en vectores)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)

        # Dropout aplicado a los embeddings
        self.emb_dropout = nn.Dropout(emb_dropout)

        # Capa LSTM empaquetada
        self.lstm = nn.LSTM(
            input_size=embed_dim,             # Entrada: embeddings
            hidden_size=hidden_dim,           # Salida: estado oculto por paso de tiempo
            num_layers=num_layers,            # Número de capas LSTM apiladas
            batch_first=True,                 # Dimensión [B, L, D] (en lugar de [L, B, D])
            dropout=dropout if num_layers > 1 else 0.0,  # Dropout solo si >1 capa
            bidirectional=bidirectional       # Activa LSTM bidireccional
        )

        # Calcula la dimensión de salida final según si es bidireccional
        self.out_dim = hidden_dim * (2 if bidirectional else 1)

    def forward(self, x, lengths):
        """
        Args:
            x: Tensor de entrada [B, L] con índices de tokens.
            lengths: Longitudes reales (sin padding) de cada secuencia [B].
        Returns:
            feats: Representaciones globales por secuencia [B, D]
        """
        # Embedding + Dropout
        emb = self.emb_dropout(self.embedding(x))  # [B, L, emb_dim]

        # Asegura que las longitudes estén en CPU (requerido por PyTorch)
        lengths_cpu = lengths.to("cpu")

        # Empaqueta las secuencias para ignorar el padding en el LSTM
        packed = pack_padded_sequence(emb, lengths_cpu, batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)

        # Desempaqueta la salida
        out, _ = pad_packed_sequence(packed_out, batch_first=True)  # [B, L_eff, D]

        # Global max pooling sobre las secuencias válidas (sin padding)
        B, L_eff, D = out.size()
        device = out.device

        # Crea una máscara para ignorar posiciones con padding
        mask = (torch.arange(L_eff, device=device)[None, :] < lengths[:, None])  # [B, L_eff]
        out_masked = out.masked_fill(~mask.unsqueeze(-1), float("-inf"))  # Padding -> -inf

        # Max pooling a lo largo de la dimensión temporal (L_eff)
        feats = out_masked.max(dim=1).values  # [B, D]

        return feats
# ================== LightningModule ==================
class CommentsClassifier(pl.LightningModule):
    def __init__(self, vocab_size, num_classes, emb_dim=128, hidden_dim=64, pad_id=0):
        super().__init__()
        self.save_hyperparameters() # Guarda todos los argumentos del constructor como hiperparámetros de Lightning (útil para reproducibilidad y logging)
        self.pad_id = pad_id
          # Bloque LSTM empaquetado para secuencias con padding
        self.lstm = LSTMBlockPacked(
            vocab_size=vocab_size,  # Tamaño del vocabulario (número total de tokens únicos)
            embed_dim=emb_dim,      # Dimensión de los embeddings
            hidden_dim=hidden_dim,  # Dimensión del LSTM
            padding_idx=pad_id,     # Token de padding
            bidirectional=True       # LSTM bidireccional
        )
        self.classifier = nn.Sequential(
            nn.Linear(self.lstm.out_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, num_classes)  # Logits crudos para clasificación multiclase
        )
        # Métricas de precisión (para entrenamiento y validación)
        self.train_acc = Accuracy(task="multiclass", num_classes=num_classes)
        self.val_acc   = Accuracy(task="multiclass", num_classes=num_classes)

    def forward(self, x):
        lengths = (x != self.pad_id).sum(dim=1)          # [B] Calcula longitudes reales (sin padding)
        feats = self.lstm(x, lengths)                    # [B, H*D] Extrae representaciones globales de las secuencias
        logits = self.classifier(feats)                  # [B, C] Pasa por la red densa
        return logits                                    # Devuelve las predicciones

    def training_step(self, batch, _):
        x, y = batch["input_ids"], batch["y"]            # y: int64 en [0..C-1]
        logits = self(x)
        loss = F.cross_entropy(logits, y)                 # Función de pérdida para clasificación multiclase
        self.train_acc(logits, y)                         # Actualiza precisión acumulada
         # Logging de métricas
        self.log("train-loss", loss, prog_bar=True, on_epoch=True)
        self.log("train-acc", self.train_acc, prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch, _=None): #Paso de validación
        x, y = batch["input_ids"], batch["y"]
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        self.val_acc(logits, y)
        self.log("val-loss", loss, prog_bar=True, on_epoch=True)
        self.log("val-acc", self.val_acc, prog_bar=True, on_epoch=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.AdamW(
            self.parameters(),
            lr=1e-3,          # Tasa de aprendizaje inicial
            weight_decay=1e-5 # Regularización L2 para evitar overfitting
            )

# ================== Instanciación segura y Trainer ==================
pad_id = vocab.get("[PAD]", 0)          # usa el PAD real de tu vocab
num_classes = 5                         # estrellas 1..5 mapeadas a 0..4
use_gpu = torch.cuda.is_available()

model = CommentsClassifier(
    vocab_size=len(vocab),               # NO sumes +1 si ya trae especiales
    num_classes=num_classes,             # Número de clases a predecir (para clasificación multiclase)
    emb_dim=128,                         # Dimensión de los vectores de embedding. puedes subir a 256 si hay memoria
    hidden_dim=64,                       # Subir a 128 o más,	mejora la capacidad del modelo para capturar dependencias complejas
    pad_id=pad_id                        # ID del token de padding, necesario para enmascarar entradas y evitar que afecten el aprendizaje.
)

tb_logger = TensorBoardLogger('tb_logs', name='LSTMClassifier')
callbacks = [EarlyStopping(monitor='val-loss', patience=7, mode='min')]

trainer = Trainer(
    accelerator="gpu" if use_gpu else "cpu",
    devices=1,
    precision="16-mixed" if use_gpu else 32,
    max_epochs=15,                        # 1 época suele quedar en ~azar; usa 3–5
    logger=tb_logger,
    callbacks=callbacks
)

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)


## Conclusiones:


El análisis mostró que la forma de representar el texto es más determinante que el modelo en sí. Al comparar enfoques, los modelos basados en TF-IDF tuvieron un desempeño claramente superior a los que usaban promedios de Word2Vec, lo que evidencia que en este dataset las frecuencias de palabras contienen la señal más valiosa.

A pesar de probar distintos algoritmos, el resultado más sólido lo dio un modelo sencillo: Logistic Regression con TF-IDF, que alcanzó una accuracy cercana al 0.605. Esto demuestra que, en este caso, los modelos simples pueden superar en eficacia a alternativas más complejas como LightGBM o redes neuronales.

Se observó también un comportamiento desigual entre clases: las clases 0 y 4 fueron predichas con buena precisión y recall, mientras que las clases 1 y 2 resultaron más difíciles de distinguir, revelando una clara asimetría en el desempeño.

En términos generales, los modelos se mantuvieron en un rango de 0.57 a 0.61 de accuracy, lo que marca un límite de rendimiento con las configuraciones probadas.

Por otro lado, la LSTM quedó muy por debajo (≈0.23 de accuracy), lo que refleja tanto las limitaciones de cómputo como la necesidad de mayor entrenamiento.

Finalmente, el intento de usar XGBoost no pudo completarse debido a la exigencia de recursos del dataset, lo que refuerza la idea de que la complejidad del problema requiere cuidado no solo en el modelado, sino también en la infraestructura.
